# Projeto Integrador (P15) - Pipeline End-to-End de Engenharia de Dados e ML

**Previsao de churn de clientes de um e-commerce**, a partir de duas fontes de
dados reais (CSV proprio + API publica do IBGE), armazenadas em PostgreSQL com
modelagem relacional, processadas por um pipeline de ETL em Python, com
engenharia de features e um modelo de classificacao treinado e avaliado.

**Fluxo**: Fontes de Dados -> PostgreSQL -> ETL Python -> Features -> Machine
Learning -> Metricas -> Resultados

Este notebook executa o pipeline completo e documenta cada decisao tecnica.
Os modulos reaproveitados (`etl/`) sao os mesmos rodados em producao via
linha de comando - nada aqui e exclusivo do notebook.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'etl'))
import pandas as pd
from sqlalchemy import create_engine

DB_URL = 'postgresql://postgres:@localhost:5432/projeto_integrador_p15'
pd.set_option('display.width', 120)

## 1. Fontes de dados

**Fonte 1 (arquivo)**: `data/clientes.csv` e `data/vendas.csv` - cadastro de
420 clientes e historico de vendas de um e-commerce (gerados sinteticamente
para esta atividade, com sujeira proposital: nulos, duplicatas e outliers,
para o ETL ter o que limpar de verdade).

**Fonte 2 (API publica, consumida ao vivo)**: `servicodados.ibge.gov.br` -
lista oficial de municipios do Brasil, usada para enriquecer cada cliente com
municipio, UF e regiao geografica oficiais do IBGE a partir do nome da
cidade cadastrada.

In [2]:
clientes_raw = pd.read_csv('../data/clientes.csv')
vendas_raw = pd.read_csv('../data/vendas.csv')
print('clientes.csv:', clientes_raw.shape)
print('vendas.csv:', vendas_raw.shape)
print()
print('Nulos em vendas.csv:')
print(vendas_raw.isnull().sum())
print('Duplicatas exatas em vendas.csv:', vendas_raw.duplicated().sum())

clientes.csv: (420, 6)
vendas.csv: (3641, 8)

Nulos em vendas.csv:
venda_id           0
cliente_id         0
produto_nome       0
categoria          0
data               0
quantidade         0
valor_unitario     0
valor_total       60
dtype: int64
Duplicatas exatas em vendas.csv: 45


In [3]:
import requests
r = requests.get('https://servicodados.ibge.gov.br/api/v1/localidades/municipios', timeout=30)
municipios = r.json()
print(f'Fonte 2 (API IBGE): {len(municipios)} municipios recebidos ao vivo')
print('Exemplo:', {k: municipios[0][k] for k in ['id', 'nome']})

Fonte 2 (API IBGE): 5571 municipios recebidos ao vivo
Exemplo: {'id': 1100015, 'nome': "Alta Floresta D'Oeste"}


## 2. ETL (Python) -> PostgreSQL

O pipeline completo (`etl/pipeline_etl.py`) faz extract (CSV + API),
transform (limpeza + enriquecimento geografico) e load (PostgreSQL), usando
o esquema relacional em `sql/schema.sql`:

- `dim_cliente`, `dim_produto`, `dim_localidade` (dimensoes)
- `fato_vendas` (fato, FK para as 3 dimensoes)

Rodado aqui dentro do notebook para registrar a execucao real.

In [4]:
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'etl'))
import pipeline_etl
pipeline_etl.run()

2026-10-01 09:40:09,043 | INFO | EXTRACT: lendo Fonte 1 (CSV) - clientes.csv e vendas.csv


2026-10-01 09:40:09,101 | INFO |   clientes.csv: 420 linhas | vendas.csv: 3641 linhas


2026-10-01 09:40:09,103 | INFO | EXTRACT: consultando Fonte 2 (API publica do IBGE, ao vivo)


2026-10-01 09:40:12,085 | INFO |   5571 municipios recebidos da API, 5290 indexados por nome


2026-10-01 09:40:12,103 | INFO | TRANSFORM: limpando vendas


...

2026-10-01 09:40:12,507 | INFO |   dim_localidade: 10 linhas


2026-10-01 09:40:12,606 | INFO |   dim_cliente: 420 linhas


2026-10-01 09:40:12,622 | INFO |   dim_produto: 18 linhas


2026-10-01 09:40:13,035 | INFO |   fato_vendas: 3296 linhas


2026-10-01 09:40:13,048 | INFO | ETL concluido com sucesso.


### Verificacao: o dado esta de fato no PostgreSQL, modelado e consultavel

In [5]:
engine = create_engine(DB_URL)
display_df = pd.read_sql('''
    SELECT l.regiao, COUNT(DISTINCT c.cliente_id) AS clientes, COUNT(f.id) AS vendas,
           ROUND(SUM(f.valor_total)::numeric, 2) AS receita
    FROM dim_cliente c
    JOIN dim_localidade l ON c.localidade_id = l.id
    JOIN fato_vendas f ON f.cliente_id = c.cliente_id
    GROUP BY l.regiao ORDER BY receita DESC
''', engine)
display_df

,regiao,clientes,vendas,receita
0,Norte,192,1502,1267658.63
1,Sudeste,128,979,791269.72
2,Nordeste,96,815,758062.08


## 3. Analise Exploratoria de Dados (EDA)

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

vendas_db = pd.read_sql('SELECT * FROM fato_vendas', engine, parse_dates=['data'])
clientes_db = pd.read_sql('SELECT * FROM dim_cliente', engine)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.histplot(vendas_db['valor_total'], bins=40, ax=axes[0], color='#4c72b0')
axes[0].set_title("Distribuicao de valor_total (apos limpeza)")
sns.countplot(y=clientes_db['segmento'], ax=axes[1], color='#55a868')
axes[1].set_title('Clientes por segmento')
plt.tight_layout()
plt.savefig('eda_1.png', dpi=100)
plt.show()

C:\Users\marce\AppData\Local\Temp\ipykernel_61888\426861060.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
vendas_cat = pd.read_sql('''
    SELECT p.categoria, COUNT(*) AS qtd_vendas, ROUND(SUM(f.valor_total)::numeric,2) AS receita
    FROM fato_vendas f JOIN dim_produto p ON f.produto_id = p.id
    GROUP BY p.categoria ORDER BY receita DESC
''', engine)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=vendas_cat, y='categoria', x='receita', ax=ax, color='#dd8452')
ax.set_title('Receita por categoria de produto')
plt.tight_layout()
plt.savefig('eda_2.png', dpi=100)
plt.show()
vendas_cat

C:\Users\marce\AppData\Local\Temp\ipykernel_61888\1512435342.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,categoria,qtd_vendas,receita
0,Eletronicos,917,1665796.75
1,Moveis,738,715339.71
2,Escritorio,497,194346.90
3,Acessorios,584,174980.75
4,Livros,560,66526.32


## 4. Engenharia de features (sem vazamento de dados)

Diferente do P14 (Modulo 5), onde a feature de recencia e o criterio do
target coincidiam de proposito (para ilustrar o problema de data leakage),
aqui o split e **temporal**: um corte de 90 dias antes da data mais recente
separa o que vira feature (tudo ate o corte) do que vira o rotulo (se o
cliente comprou DEPOIS do corte). Isso evita que o modelo "veja" informacao
do futuro.

In [8]:
import feature_engineering as fe
clientes_fe, vendas_fe = fe.carregar_dados(engine)
features, corte, data_max = fe.construir_features(clientes_fe, vendas_fe)

print(f'Data mais recente nas vendas: {data_max.date()}')
print(f'Corte (90 dias antes): {corte.date()}')
print(f'Clientes com features: {len(features)}')
print(f'Taxa de churn: {features["churn"].mean():.2%}')
print(f'Total de features: {features.shape[1] - 2}')
features.to_csv('../data/features.csv', index=False)
features.head()

Data mais recente nas vendas: 2026-10-01
Corte (90 dias antes): 2026-07-03
Clientes com features: 392
Taxa de churn: 55.61%
Total de features: 18


,cliente_id,total_compras,qtd_compras,ticket_medio,dias_desde_ultima_compra,dias_desde_primeira_compra,categorias_distintas,dias_desde_cadastro,churn,segmento_Premium,segmento_Regular,segmento_VIP,regiao_Nordeste,regiao_Norte,regiao_Sudeste,categoria_preferida_Acessorios,categoria_preferida_Eletronicos,categoria_preferida_Escritorio,categoria_preferida_Livros,categoria_preferida_Moveis
0,1,3942.47,7,563.210000,81,486,3,494,1,False,True,False,False,False,True,False,False,False,False,True
1,2,14683.79,13,1129.522308,67,340,5,585,1,False,False,True,False,True,False,False,True,False,False,False
2,3,15980.81,10,1598.081000,100,451,4,556,0,False,True,False,True,False,False,False,True,False,False,False
3,4,1492.21,3,497.403333,63,508,3,570,1,False,True,False,False,False,True,False,False,False,False,True
4,5,5080.56,4,1270.140000,4,49,3,52,0,True,False,False,False,True,False,False,True,False,False,False


In [9]:
fig, ax = plt.subplots(figsize=(8, 6))
cols_corr = ['total_compras', 'qtd_compras', 'ticket_medio', 'dias_desde_ultima_compra',
             'dias_desde_primeira_compra', 'categorias_distintas', 'dias_desde_cadastro', 'churn']
sns.heatmap(features[cols_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlacao entre as features numericas e o target (churn)')
plt.tight_layout()
plt.savefig('eda_3.png', dpi=100)
plt.show()

C:\Users\marce\AppData\Local\Temp\ipykernel_61888\3617216695.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretacao**: ao contrario do P14, nenhuma feature tem correlacao
artificialmente perfeita com o target - `dias_desde_ultima_compra` (recencia
DENTRO da janela de observacao) e a mais forte, o que faz sentido de
negocio (quem ja estava inativo antes do corte tende a continuar inativo),
mas nao e uma tautologia como antes, porque o target olha para um periodo
que as features nunca viram.

### Por que cada feature foi criada

- **total_compras / qtd_compras / ticket_medio**: valor, frequencia e tiquete
  medio do cliente ate o corte - as tres metricas classicas de valor de
  cliente (RFM sem o R).
- **dias_desde_ultima_compra**: recencia dentro da janela de observacao -
  proxy de engajamento recente sem olhar para o futuro.
- **dias_desde_primeira_compra**: ha quanto tempo o cliente comprou pela
  primeira vez (tempo de relacionamento efetivo, diferente de data de
  cadastro).
- **categorias_distintas**: diversidade de consumo.
- **dias_desde_cadastro**: maturidade do cadastro.
- **segmento_* (one-hot)**: categoria declarada do cliente (Premium/Regular/
  VIP) - sem ordem natural, por isso one-hot em vez de encoding ordinal.
- **regiao_* (one-hot)**: regiao geografica oficial do IBGE (Fonte 2) -
  mostra o enriquecimento da segunda fonte de dados virando feature real.
- **categoria_preferida_* (one-hot)**: categoria de produto onde o cliente
  mais gastou ate o corte.

## 5. Treinamento e comparacao de modelos

In [10]:
import treinar_modelo as tm
df_res, relatorios, (X_test, y_test) = tm.treinar(features)

Logistic Regression
accuracy=0.6582  f1=0.6966  roc_auc=0.6565  cv_f1_mean=0.7169
              precision    recall  f1-score   support

           0       0.62      0.60      0.61        35
           1       0.69      0.70      0.70        44

    accuracy                           0.66        79
   macro avg       0.65      0.65      0.65        79
weighted avg       0.66      0.66      0.66        79



Random Forest
accuracy=0.7595  f1=0.7912  roc_auc=0.8185  cv_f1_mean=0.7609
              precision    recall  f1-score   support

           0       0.75      0.69      0.72        35
           1       0.77      0.82      0.79        44

    accuracy                           0.76        79
   macro avg       0.76      0.75      0.75        79
weighted avg       0.76      0.76      0.76        79



Gradient Boosting
accuracy=0.7215  f1=0.7500  roc_auc=0.8357  cv_f1_mean=0.7267
              precision    recall  f1-score   support

           0       0.69      0.69      0.69        35
           1       0.75      0.75      0.75        44

    accuracy                           0.72        79
   macro avg       0.72      0.72      0.72        79
weighted avg       0.72      0.72      0.72        79

RANKING DE MODELOS (por F1):
             modelo  accuracy       f1  roc_auc  cv_f1_mean
      Random Forest  0.759494 0.791209 0.818506    0.760920
  Gradient Boosting  0.721519 0.750000 0.835714    0.726718
Logistic Regression  0.658228 0.696629 0.656494    0.716854



Modelo salvo: Random Forest (F1=0.7912) -> models/modelo_churn.pkl


In [11]:
fig, ax = plt.subplots(figsize=(7, 4))
df_plot = df_res.melt(id_vars='modelo', value_vars=['accuracy', 'f1', 'roc_auc'])
sns.barplot(data=df_plot, x='modelo', y='value', hue='variable', ax=ax)
ax.set_title('Comparacao de metricas entre os 3 modelos')
ax.set_ylim(0, 1)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('eda_4.png', dpi=100)
plt.show()

C:\Users\marce\AppData\Local\Temp\ipykernel_61888\2463675978.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Conclusoes

- O pipeline roda de ponta a ponta: duas fontes de dados reais (CSV +
  API publica do IBGE, consumida ao vivo) -> PostgreSQL com modelagem
  relacional (3 dimensoes + 1 fato) -> ETL com limpeza documentada (IQR
  por categoria, nulos, duplicatas) -> 18 features (temporais, agregacoes,
  encoding) -> split temporal sem vazamento -> comparacao de 3 modelos ->
  modelo salvo em `.pkl`.
- O **Random Forest** venceu por F1 (0.79) e tambem teve a melhor
  generalizacao em cross-validation (cv_f1_mean=0.76, proximo do F1 de
  teste - sinal de que nao esta overfitando). O Gradient Boosting teve o
  melhor ROC-AUC (0.84), indicando boa separacao de classes mesmo com F1
  levemente menor.
- Diferente do exercicio do Modulo 5, estes resultados sao **o quanto o
  modelo realmente consegue prever churn usando so informacao disponivel
  antes do corte temporal** - nao ha vazamento da definicao do target para
  dentro das features.
- Proximos passos razoaveis para evoluir o projeto: tunar hiperparametros
  (GridSearch/RandomizedSearch), testar uma janela de observacao/label
  diferente, e agregar uma terceira fonte de dados (ex.: indicadores
  socioeconomicos do IBGE por regiao) para enriquecer ainda mais as
  features geograficas.